# Given best hyper-params, how much variance does each impactme outcome explain?

- Variance explained will not be disjoint

In [ ]:
import os
import sys

sys.path.append(os.path.abspath("../"))
from src.config import BASE_PATH
from src.utils import IMPACTME_OUTCOMES_SHOT, IMPACTME_OUTCOMES_LR

import pandas as pd
import numpy as np
from src.swarm import run_outcome_swarm, make_full_lr, make_full_shot

In [ ]:
layer_list = [2, 3]
impact_list = ["self", "parent"]
agg_list = [
    "block",
    "max",
    "avg",
    "proba-limited",
    "binary",
]
linear_types = ["elastic", "ridge"]
swarm_dir = BASE_PATH / "swarm_dir" / "outcome_analysis"
N_JOBS = 4
N_THREADS = 2
TUNE_FLAG = "True"
# impactme_outcomes = IMPACTME_OUTCOMES
# impactme_outcomes=['f3']

In [ ]:
full_cmd_list = []
for agg_type in agg_list:
    if "shot" in agg_type:
        impactme_outcomes = IMPACTME_OUTCOMES_SHOT
    else:
        impactme_outcomes = IMPACTME_OUTCOMES_LR  # doesnt have a8,e3,e4
    swarm_path = swarm_dir / "commands" / f"{agg_type}.swarm"
    if swarm_path.exists():
        swarm_path.unlink()
    swarm_path.parent.mkdir(exist_ok=True, parents=True)
    cmd_list = []
    for impact_type in impact_list:
        for linear_type in linear_types:
            for n_layers in layer_list:
                param_path = (
                    BASE_PATH
                    / "initial_results"
                    / "combined_folds"
                    / f"{n_layers}_layers"
                    / linear_type
                    / "param_summary"
                    / "r2"
                    / f"{impact_type}.tsv"
                )
                param_df = pd.read_csv(param_path, sep="\t", index_col=0)
                if agg_type in ["proba-limited", "binary"]:
                    agg_search = f"0_shot_{agg_type}"
                else:
                    agg_search = agg_type
                param_sub = param_df[param_df["agg"] == agg_search].sort_values(
                    by="avg_r2", ascending=False
                )
                best_row = param_sub.iloc[0]
                model_name = best_row["model"]
                desc_seg = best_row["desc_seg"]
                ## Only matters for 3 layers, but can leave as None
                ## Don't loop through possible params, only ones specified here
                # l1_ratio
                l1_2 = best_row["l1_2"]
                # also change "None" string --> None
                l1_2 = None if (pd.isna(l1_2) or str(l1_2).strip() == "None") else l1_2
                l1_list = [l1_2] if l1_2 is not None else None
                # alpha
                alpha_2 = best_row["alpha_2"]
                alpha_2 = (
                    None
                    if (pd.isna(alpha_2) or str(alpha_2).strip() == "None")
                    else alpha_2
                )
                alpha_list = [alpha_2] if alpha_2 is not None else None
                BASE_SAVE_DIR = (
                    BASE_PATH / "outcome_analysis" / f"{n_layers}_layers" / linear_type
                )
                if agg_type in ["block", "max", "avg"]:
                    lr_cmd_list = make_full_lr(
                        seg_types=[desc_seg],
                        model_name=model_name,
                        impact_type=impact_type,
                        agg_type=agg_type,
                        base_save_dir=BASE_SAVE_DIR,
                        n_jobs=N_JOBS,
                        n_threads=N_THREADS,
                        tune_flag_str=TUNE_FLAG,
                        n_layers=n_layers,
                        l1_list=l1_list,
                        alpha_list=alpha_list,
                        impactme_outcomes=impactme_outcomes,
                        linear_type=linear_type,
                        metric_agg="combined_folds",
                    )
                    cmd_list += lr_cmd_list
                else:
                    # remove desc_ prefix
                    desc = "_".join(desc_seg.split("_")[1:])
                    cmd_str = make_full_shot(
                        model_name=model_name,
                        impact_type=impact_type,
                        temp=0.6,
                        pred_type=agg_type,
                        desc_type=desc,
                        base_save_dir=BASE_SAVE_DIR,
                        n_jobs=N_JOBS,
                        n_threads=N_THREADS,
                        tune_flag_str=TUNE_FLAG,
                        n_layers=n_layers,
                        l1_list=l1_list,
                        alpha_list=alpha_list,
                        impactme_outcomes=impactme_outcomes,
                        linear_type=linear_type,
                        metric_agg="combined_folds",
                    )
                    cmd_list.append(cmd_str)
    swarm_path.write_text("\n".join(cmd_list))
    full_cmd_list += cmd_list
print(f"Num commands: {len(full_cmd_list)}")

Run sequentially locally

In [ ]:
# import subprocess

# for iteration, cmd in enumerate(full_cmd_list):
#     print(f"{iteration +1}/{len(full_cmd_list)}...")
#     print(cmd)
#     subprocess.run(cmd, shell=True, check=True)
#     break

Run swarm

In [ ]:
run_outcome_swarm(
    swarm_log_dir=swarm_dir / "logs",
    cmd_dir=swarm_dir / "commands",
    agg_types=agg_list,
)